# VoxCPM2 模型在 Kaggle GPU (T4 x2) 上生成音频测试

## 关键修正 (V5)
- ✅ T4 x2 加速器 (`--accelerator NvidiaTeslaT4`)
- ✅ 模型下载: requests 手动下载 hf-mirror/openbmb/VoxCPM2 (绕过 HF hub HEAD bug, 9/9 文件含 4.58GB)
- ✅ 加载: 官方 `voxcpm` 库 `VoxCPM.from_pretrained(本地路径)` (非 FunASR)
- ✅ 不硬钉 torch, 让 pip 拉 ≥2.5.0 兼容版
- ✅ 生成后写 .wav + RTF 测算

In [ ]:
# === 在任何 import 之前设 HF 镜像 endpoint (huggingface_hub 导入时固化) ===
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
os.environ["HF_HUB_DISABLE_SSL_VERIFY"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"

import sys
import subprocess

# 安装官方推理库 voxcpm (README: pip install voxcpm)
# voxcpm 依赖 torch>=2.5.0; 不硬钉版本, 让 pip resolver 选 Kaggle 兼容组合
print("Installing voxcpm + deps...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir",
    "voxcpm==2.0.3",
], check=True)
print("Dependencies installed.")
import torch
print(f"torch={torch.__version__}, cuda={torch.cuda.is_available()}")

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"CUDA version: {torch.version.cuda}")
    print(f"cuDNN: {torch.backends.cudnn.version()}")
else:
    print("CUDA not available - 请在 Kaggle 设置中开启 GPU")

In [ ]:
# 下载 VoxCPM2 模型 (绕过 huggingface_hub 库, 用 requests 手动下载)
# huggingface_hub.snapshot_download 的 HEAD 元数据失败 (FileMetadataError),
# 改用 requests 直接访问 hf-mirror.com/openbmb/VoxCPM2/resolve/main/<file>,
# 打印每个文件真实 HTTP 状态码, 彻底定位是 401/403/404 还是网络问题
import os, sys
import requests
from huggingface_hub import HfApi

os.environ.setdefault("HF_ENDPOINT", "https://hf-mirror.com")
ENDPOINT = "https://hf-mirror.com"
REPO = "openbmb/VoxCPM2"  # 诊断确认存在
model_dir = "/kaggle/working/VoxCPM2"
os.makedirs(model_dir, exist_ok=True)

# 1) 拿文件列表
api = HfApi(endpoint=ENDPOINT)
info = api.model_info(REPO)
files = sorted(s.rfilename for s in info.siblings)
print(f"repo {REPO} 文件列表 ({len(files)}):")
for f in files:
    print(f"  - {f}")
print(f"private={info.private}, gated={getattr(info, 'gated', None)}")

print(f"\n=== 逐文件下载 (base={ENDPOINT}) ===")
ok = 0
for fname in files:
    url = f"{ENDPOINT}/{REPO}/resolve/main/{fname}"
    dest = os.path.join(model_dir, fname)
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    try:
        r = requests.get(url, timeout=120, stream=True, allow_redirects=True,
                         verify=False)
        print(f"{fname}: HTTP {r.status_code} | {len(r.content) if r.status_code==200 else r.reason}")
        if r.status_code == 200:
            with open(dest, "wb") as f:
                for chunk in r.iter_content(8192):
                    f.write(chunk)
            sz = os.path.getsize(dest)
            print(f"   ✅ saved {sz/1e6:.2f} MB")
            ok += 1
        elif r.status_code in (401, 403):
            print(f"   ❌ 需认证 (gated/private), 跳过")
    except Exception as e:
        print(f"{fname}: ERR {type(e).__name__}: {str(e)[:100]}")

print(f"\n下载完成: {ok}/{len(files)} 文件成功")
print("目录内容:", os.listdir(model_dir))

In [ ]:
# 检查模型结构
import os
model_path = "/kaggle/working/VoxCPM2"
for root, dirs, files in os.walk(model_path):
    depth = root.replace(model_path, '').count(os.sep)
    if depth <= 2:
        for f in files:
            print(os.path.join(root, f))

In [ ]:
# 用官方 voxcpm 库加载 VoxCPM2 (本地已下载到 /kaggle/working/VoxCPM2)
# 关键: from_pretrained 若传本地目录则直接复用, 不重新下载
# load_denoiser=False 跳过额外 ModelScope 降噪模型下载
import time, sys
from voxcpm import VoxCPM

model_path = "/kaggle/working/VoxCPM2"  # cell 3 已下载
print(f"Loading VoxCPM2 from local: {model_path}")
t0 = time.time()
model = VoxCPM.from_pretrained(
    model_path,
    load_denoiser=False,
    optimize=False,   # 调试: 关掉 torch.compile warmup, 避免耗时
    device="cuda",
)
print(f"模型加载完成, 耗时 {time.time()-t0:.1f}s")
print(f"采样率: {model.tts_model.sample_rate} Hz")

In [ ]:
# VoxCPM2 推理测试 (官方 generate API)
import time, os
import numpy as np
import soundfile as sf

sr = model.tts_model.sample_rate
texts = [
    "Hello, this is a test of VoxCPM2 text to speech synthesis.",
    "你好，这是 VoxCPM2 模型生成的中文语音测试。",
    "The quick brown fox jumps over the lazy dog.",
]

print(f"Starting generation (sr={sr})...")
total_rtf = 0.0
for i, text in enumerate(texts):
    print(f"\nTest {i+1}: {text[:50]}")
    try:
        t0 = time.time()
        wav = model.generate(
            text=text,
            cfg_value=2.0,
            inference_timesteps=10,
        )
        t1 = time.time()
        synth = t1 - t0
        wav = np.asarray(wav).astype(np.float32).reshape(-1)
        dur = len(wav) / sr
        rtf = synth / dur if dur > 0 else 0
        total_rtf += rtf
        out = f"/kaggle/working/voxcpm2_test_{i}.wav"
        sf.write(out, wav, sr)
        print(f"   ✅ 时长 {dur:.2f}s | 合成 {synth:.2f}s | RTF {rtf:.3f} | {out}")
    except Exception as e:
        print(f"   ❌ {type(e).__name__}: {e}")
        import traceback; traceback.print_exc()

if total_rtf > 0:
    print(f"\n平均 RTF: {total_rtf/len(texts):.3f}")

In [ ]:
# 验证生成的音频文件 + 显存占用
import os
import soundfile as sf
import torch
print("=== 生成的音频文件 ===")
for f in sorted(os.listdir("/kaggle/working")):
    if f.startswith("voxcpm2_test_") and f.endswith(".wav"):
        path = f"/kaggle/working/{f}"
        wav, sr = sf.read(path)
        dur = len(wav) / sr
        sz = os.path.getsize(path) / 1e6
        print(f"{f}: {dur:.2f}s, {sr}Hz, {sz:.2f}MB")
if torch.cuda.is_available():
    alloc = torch.cuda.memory_allocated() // 1024 // 1024
    total = torch.cuda.get_device_properties(0).total_memory // 1024 // 1024
    print(f"\n显存: {alloc}/{total} MB")
print("\n=== 测试完成, 请到 Kaggle 文件面板下载 .wav 试听 ===")

## 结果分析

1. **模型加载**: 检查模型是否正确加载到 GPU
2. **推理速度**: 观察 RTF (Real-Time Factor = 合成时间 / 音频时长)
3. **显存占用**: 观察 GPU 显存占用
4. **音频质量**: 听取生成的音频质量

## 预期指标 (VoxCPM2 T4/V100 FP16):
- RTF (实时率): ~0.05-0.1 (即 1秒音频需 0.05-0.1 秒合成)
- 显存占用: ~10-12 GB (FP16) / ~8-10 GB (INT8)
- 音频采样率: 24kHz
- 支持: 零样本音色克隆、语速控制、情感控制

## ⚠️ Kaggle 限制提醒
- Session 最长 12 小时，周配额 30 小时
- 无持久公网 IP/URL，不适合生产 HA
- 每次重启需重新下载模型、加载模型
- 生产环境建议使用 Modal (A10G/V100) + 固定端点